In [8]:
import math
import numpy as np
import torch
import warnings

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from moudel.mgmcformer import MgMcFORMER
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from collections import Counter
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator
from data.preprocessing import toDataloader
from data.preprocessing import toDataloader_1
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from glob import glob
from astropy.io import fits
import pywt
from PCA_host_substract import fit_spectrum,fit_spectrum_with_sn_template

warnings.filterwarnings("ignore")





def clip_wavelength_range(wave, flux, lambda_min, lambda_max):
    mask = (wave >= lambda_min) & (wave <= lambda_max)
    return wave[mask], flux[mask]

def wavelet_denoise(flux, wavelet='db4', level=2, threshold_scale=1.0):
    
    
    coeffs = pywt.wavedec(flux, wavelet=wavelet, level=level)
    
    
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = threshold_scale * sigma * np.sqrt(2 * np.log(len(flux)))

    
    new_coeffs = [coeffs[0]]  
    for c in coeffs[1:]:
        new_c = pywt.threshold(c, threshold, mode='soft')
        new_coeffs.append(new_c)

    
    denoised_flux = pywt.waverec(new_coeffs, wavelet)
    
    
    return denoised_flux[:len(flux)]




def process_spectrum_to_3channel(flux_wave, flux_val,
                                 lambda_min=4000, lambda_max=7000,
                                 target_len=4000,
                                 degree=3, sigma_clip=3.0,
                                 apply_wavelet=True,
                                 wavelet='db4', level=2, threshold_scale=1.0,
                                 PCA_substract = False,
                                 filename=""):
       
    valid =  np.isfinite(flux_val)
    if np.sum(valid) < 10:
        return np.zeros((target_len, 2), dtype=np.float32)

    wave = flux_wave[valid]
    flux = flux_val[valid]


    max_flux = np.max(flux)
    if max_flux <= 0 or np.isnan(max_flux):
        return np.zeros((target_len, 2), dtype=np.float32)
    norm_flux = flux / max_flux


    if apply_wavelet:
        norm_flux = wavelet_denoise(norm_flux, wavelet=wavelet, level=level, threshold_scale=threshold_scale)
    
    
    if PCA_substract == True:
        
    
       
        lower_limit, upper_limit = 3999, 7001

       
        mask = (wave >= lower_limit) & (wave <= upper_limit)
        wave = wave[mask]
        norm_flux = norm_flux[mask]

        # 更新 data
        data = [wave, norm_flux]
        
        host_template_path = './PCA_run/meansk86.dat'
        
        
        sn_star_path = "./PCA_run/pca_flux_star.txt"
        Total_path  = "./PCA_run/pca_flux_total.txt"

        
        
        
        coeffs_Total, fit_model, residual, residual_rms_IIb, sn_model = fit_spectrum_with_sn_template(
            data,
            host_template_path,
            sn_template_path=Total_path,
            sn_n_components=20,
           
            denoise=True,
            visualize=False,
            plot_sn_component=True,   
            wavelet="db4"
        )
        
        

        coeffs_star, fit_model, residual, residual_rms_star, sn_model = fit_spectrum_with_sn_template(
            data,
            host_template_path,
            sn_template_path=sn_star_path,
            sn_n_components=20,
            transient_only=True,
            denoise=True,
            visualize=False,
            plot_sn_component=True,  
            wavelet="db4"
        )
        
       
        
    
    return np.stack([coeffs_Total,coeffs_star
                     ])
    


In [2]:
def corssmatch_catalog(catalog_file, source_name):
    import pandas as pd

    
    df = pd.read_csv(catalog_file)

    
    if source_name not in df['TARGETID'].values:
        return f"Source {source_name} Not in TARGETID .", False

    
    matched = df[df['TARGETID'] == source_name]

   
    valid = matched[(matched['ZWARN'] == 0) & (matched['OBJTYPE'] == 'TGT')]

    if not valid.empty:
        return f"Source {source_name} exists，ZWARN=0 and OBJTYPE=TGT。", True
    else:
        return f"Source {source_name} exists， ZWARN!=0 or OBJTYPE!=TGT。", False



class SDSSCatalogMatcher_0:
    def __init__(self, catalog_file):
        
        self.df = pd.read_csv(catalog_file)
        self.df.set_index('TARGETID', inplace=True, drop=False)  

    def match_source(self, source_name):
       
        if source_name not in self.df.index:
            return False, f"Source {source_name} not in TARGETID."

        row = self.df.loc[source_name]

       
        if isinstance(row, pd.DataFrame):
            valid = row[(row['ZWARN'] == 0) & (row['OBJTYPE'] == 'TGT')]
        else:
            valid = row if (row['ZWARN'] == 0 and row['OBJTYPE'] == 'TGT') else None

        if valid is not None and not (isinstance(valid, pd.DataFrame) and valid.empty):
            return True, f"Source {source_name} exists，ZWARN=0 and OBJTYPE=TGT。"
        else:
            return False, f"⚠Source {source_name} exists， ZWARN!=0 or OBJTYPE!=TGT。"

In [4]:

def metadata_check(filename):
    
    hdul = fits.open(filename)
    header = hdul[0].header
    zwarn = hdul[0].header['Z_WARNIN']
    redshift = hdul[0].header['Z']
    data = hdul[0].data
    objtype = hdul[0].header['OBJTYPE']
    
    loglam = header['COEFF0'] + header['COEFF1'] * np.arange(header['NAXIS1'])
    wavelength = 10 ** loglam

    wavelength = wavelength/(1+redshift)
    wave = wavelength
    lower_limit, upper_limit = 3900, 7100

   
    mask = (wave >= lower_limit) & (wave <= upper_limit)
    wave = wave[mask]
    
    if len(data[0])<1000 or len(wave)<1000:
        return False
    
    if objtype!='GALAXY':
        #if filename == "/share/data/WideF-S/SDSS/dr7/spectro/ss_tar_26/0271/spSpec/spSpec-51883-0271-171.fit":
        #        print('test-171: -2 objtype False')
        return False
    
    
   
    
    if np.min(wave)>4100 or np.max(wave)<6900:
        #if filename == "/share/data/WideF-S/SDSS/dr7/spectro/ss_tar_26/0271/spSpec/spSpec-51883-0271-171.fit":
        #        print('test-171: -2 wave range False')
        return False
    
    flux        = data[0]
    error       = data[2]
    
    #wavelength = wavelength[(mask == 0) ]
    #flux = flux[(mask == 0)]
    #error = error[(mask == 0)]
    
    
    
    with np.errstate(divide='ignore', invalid='ignore'):
        snr = np.where((error > 0) & (error != 0), np.abs(flux) / (error), 0)
    median_snr = np.median(snr[snr > 0]) if np.any(snr > 0) else 0
    
    if median_snr<15:
        #print('Low SNR')
        return False
    
    #print(np.min(wave),np.max(wave))
    return True

def interpolate_flux(wave_src, flux_src, wave_target):
    interp_func = interp1d(wave_src, flux_src, bounds_error=False,
                           fill_value=(flux_src[0], flux_src[-1]))
    return interp_func(wave_target)


def read_SDSS_Spectrum_bin(filename):
    from astropy.io import fits
    import numpy as np
    import matplotlib.pyplot as plt
    
    hdul = fits.open(filename)
    data = hdul[0].data
    
    

    flux        = data[0] 
    continuum   = data[1]  
    error       = data[2]  
    mask        = data[3] 

    
    header = hdul[0].header
    
    zwarn = hdul[0].header['Z_WARNIN']
    
    ra = hdul[0].header['RAOBJ']
    dec = hdul[0].header['DECOBJ']
    
    print('RA = ',ra)
    print('DEC = ',dec)
    

    redshift = header.get('Z',0)
    
    fiber = hdul[0].header['FIBERID']
    plate = hdul[0].header['PLATEID']
    mjd = hdul[0].header['MJD']
    
    objtype = hdul[0].header['OBJTYPE']

    loglam = header['COEFF0'] + header['COEFF1'] * np.arange(header['NAXIS1'])
    wavelength = 10 ** loglam

    wavelength = wavelength/(1+redshift)
   
    
    with np.errstate(divide='ignore', invalid='ignore'):
        snr = np.where((error > 0) & (error != 0), np.abs(flux) / (error), 0)
    median_snr = np.median(snr[snr > 0]) if np.any(snr > 0) else 0
    
    
    lambda_min = 4000
    lambda_max = 7000
    target_len = 4000
    
    wave_flag = True
    
    if len(wavelength)==0:
        wave_flag = False
        
    else:
        if np.max(wavelength)<6800 or np.min(wavelength)>4200:
            wave_flag = False
    

    #wavelength, flux = clip_wavelength_range(wavelength, flux, lambda_min, lambda_max)
    
    if len(wavelength)>0:
        flux_max = np.max(flux)
    else:
        flux_max = 1
        wave_flag = False

    valid_flux = flux[flux != 0]
    median_flux = np.median(valid_flux) if len(valid_flux) > 0 else 0
    #flux = flux / median_flux if median_flux > 0 else flux
    
    flux = flux/flux_max
    

    
    
    
    if wave_flag == True and median_snr>10:
        processed = process_spectrum_to_3channel(
            wavelength, flux,
            lambda_min=lambda_min,
            lambda_max=lambda_max,
            target_len=target_len,
            filename=filename,
            PCA_substract = True,
            apply_wavelet=True
        )
        
        
        
    else:
        processed = []
    
    
    if len(processed)==0:
            wave_flag = False
   
  
    
    final_data = np.expand_dims(processed, axis=0)
    
    
    
    return final_data, median_snr, wave_flag,redshift,zwarn,fiber,plate,mjd,objtype



    



In [5]:
from astropy.io import fits
import os
import pandas as pd

import os
import pandas as pd
from astropy.io import fits

def extract_sdss_key_from_path(filename):
    """
    such as 'spec-0266-51630-0001'
    """
    basename = os.path.basename(filename)
    if basename.startswith("spec-") and basename.endswith(".fits"):
        return basename.replace(".fits", "")
    else:
        raise ValueError(f"Invalid SDSS filename format: {basename}")
        
        
class SDSSCatalogMatcher:
    def __init__(self, catalog_path):
        import pandas as pd

        self.catalog = pd.read_csv(catalog_path)
        self.catalog['PLATE'] = self.catalog['PLATE'].astype(int)
        self.catalog['MJD'] = self.catalog['MJD'].astype(int)
        self.catalog['FIBERID'] = self.catalog['FIBERID'].astype(int)

        
        self.lookup = {}
        for _, row in self.catalog.iterrows():
            key = (row['PLATE'], row['MJD'], row['FIBERID'])
            self.lookup[key] = row.to_dict()

    def extract_key_from_filename(self, filename):

        
        import os
        basename = os.path.basename(filename)
        if basename.endswith('.fit'):
            parts = basename.replace('.fit', '').split('-')
            if len(parts) == 4 and parts[0] == 'spSpec':
                plate = int(parts[2])
                mjd = int(parts[1])
                fiberid = int(parts[3])
                return (plate, mjd, fiberid)
        raise ValueError(f"Invalid SDSS filename format: {filename}")

    def get_entry_by_filename(self, filename):
        
        key = self.extract_key_from_filename(filename)
        return self.lookup.get(key, None)
        
        
        


In [12]:
def mean_standardize_fit(X):
    m1 = np.mean(X, axis=1)
    mean = np.mean(m1, axis=0)

    s1 = np.std(X, axis=1)
    std = np.mean(s1, axis=0)

    return mean, std


def mean_standardize_transform(X, mean, std):
    return (X - mean) / std


def preprocess(X_train,mean,std):
    mean, std = mean,std
    X_train = mean_standardize_transform(X_train, mean, std)

    X_train_task = torch.as_tensor(X_train).float()
    

    return X_train_task

def load_model(model_name = "test_model_ZTF.pth",model_type='normal'):
    import argparse
    import utils
    
    
    if model_type == 'normal':
      
        
        #model PCA SN temp 1
        mean_1 =[-9.14157992e-02, -4.84763106e-02, -1.21939969e-02,  9.42267097e-03,
        9.91551364e-03,  6.12955221e-03,  6.15144325e-04, -8.25801876e-05,
        3.86918987e-03, -8.29895631e-04,  3.80708952e-02, -1.01873039e-01,
        2.25995740e-03, -1.35196525e-01, -1.81868073e-03, -1.53022624e-04,
       -2.36580664e-03,  2.65498365e-03,  5.22293324e-05, -3.48398653e-02,
       -6.76419780e-03, -2.87163704e-02, -3.99245183e-02,  4.03884744e-03,
       -4.43283350e-02, -1.72795987e-02, -4.14176212e-03,  4.17812363e-05,
        1.10280789e-05,  5.48719730e-03, -3.98125402e-03, -7.85933083e-03,
        1.96545589e-02,  4.30188754e-04, -1.70947019e-02,  2.94995318e-06,
        1.18769427e-01, -6.05501932e-04, -4.57909638e-04, -4.88704603e-03,
        4.61343982e-04,  2.34623520e-03, -9.44881494e-03,  3.89983287e-03,
       -3.31890716e-04, -4.16305854e-03, -4.87559308e-02,  2.62791999e-03,
        4.86241764e-01, -4.08976682e-02, -8.94292149e-03,  2.95907211e-04,
        3.67533741e-03, -4.85122397e-03,  1.28235959e-03,  3.37617370e-02,
        1.15437387e-04, -5.85002232e-02, -9.80371467e-04,  1.50530419e-01,
       -5.91459376e-03,  5.14322171e-02,  2.56738732e-03, -4.93739505e-03,
        6.14455811e-03, -2.26770418e-04, -5.43045153e-03, -1.22849737e-01,
        4.77522331e-02,  3.56975846e-04,  1.06203401e-02, -9.89298537e-03,
       -1.98207853e-04,  5.51565862e-03,  5.38984568e-02, -1.36287619e-02,
        2.10598552e-03,  3.55344008e-04,  5.89393863e-02,  3.42875242e-03,
       -7.78722419e-03, -2.33499555e-03, -2.08018090e-04, -3.05730015e-03,
       -9.45059591e-04, -3.05944433e-03,  9.74802970e+00,  3.51351352e+00,
        7.39987869e-01,  1.27908291e+00,  2.18462266e-01, -6.20148385e-01,
       -6.90529337e-01, -1.82576408e-01, -2.78220887e-01, -3.24922512e-01,
       -3.29092243e-01,  2.09947759e-01,  1.62376136e-01, -3.72169000e-02,
       -1.03897378e-01,  5.84078529e-02, -6.28449598e-02,  1.96433100e-01,
        3.04103474e-01,  4.32767272e-02]
        
        std_1 = [4.05834365e-01, 1.51859837e-01, 1.54058845e-01, 2.58144999e-02,
       6.44049860e-02, 8.42989518e-02, 6.09604929e-03, 3.15860885e-03,
       1.44363161e-02, 4.73250336e-03, 1.04725953e-01, 4.04971781e-01,
       1.49785910e-02, 4.68855295e-01, 1.87614704e-02, 1.71514332e-02,
       1.23921428e-02, 2.33319670e-02, 1.53864789e-03, 6.08008055e-02,
       2.92654344e-02, 6.71379541e-01, 2.78911799e-01, 1.73820219e-02,
       4.90219610e-01, 3.75886602e-02, 3.70322208e-02, 2.34399541e-03,
       3.56965941e-03, 2.12573275e-02, 1.85121532e-02, 3.12084888e-02,
       9.50404336e-02, 4.65824647e-03, 1.93766269e-01, 3.20256969e-03,
       3.75175097e-01, 5.61679703e-03, 3.67656272e-03, 1.64214604e-02,
       4.37600014e-03, 2.75853116e-02, 4.44590477e-02, 3.14360732e-02,
       4.87004081e-03, 1.61425837e-02, 8.10298819e-02, 2.77563944e-02,
       7.45588790e-01, 3.82790240e-01, 7.49917873e-02, 5.90791543e-03,
       1.01119844e-01, 2.01542525e-02, 1.78246979e-02, 1.63699067e-01,
       4.30214771e-03, 1.67664340e-01, 5.83522826e-03, 3.89991257e-01,
       2.44372762e-02, 2.01276566e-01, 1.27322681e-02, 2.64969406e-02,
       4.14998472e-02, 4.51078498e-03, 2.13688457e-02, 2.35984886e-01,
       1.43967377e-01, 2.56892130e-03, 3.67478599e-02, 4.69183552e-02,
       1.57505270e-03, 2.01404603e-02, 1.21343903e-01, 3.61425586e-02,
       1.37302568e-02, 9.69267083e-03, 1.19993708e-01, 7.01379400e-02,
       2.35975209e-02, 6.85428534e-03, 1.42536811e-03, 5.81643420e-02,
       9.21919646e-03, 1.34823647e-02, 9.54702069e+00, 3.68048893e+00,
       8.97245494e-01, 1.32790292e+00, 2.38308240e-01, 7.20409640e-01,
       7.13796182e-01, 2.34184255e-01, 2.56476150e-01, 4.16166139e-01,
       3.26698430e-01, 2.15676799e-01, 2.00915907e-01, 8.29707103e-02,
       1.41473322e-01, 1.03093124e-01, 1.09595226e-01, 2.25615698e-01,
       3.40694338e-01, 6.67255127e-02]
        
    
  
    
    
    
   
    # some lines here are alomst useless, but delect them will make mistake.
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', type=str, default='Training_Set_27F_PCA_SNmod_mock-1')
    parser.add_argument('--multi_group', type=list, default=[1,2],
                    help='Input list')  # group<=math.ceil(sqrt(seq_len))
    parser.add_argument('--batch', type=int, default=1, help='Dataset batch')
    parser.add_argument('--lr', type=float, default=0.0001)
    parser.add_argument('--nlayers', type=int, default=2)
    parser.add_argument('--emb_size', type=int, default=128)
    parser.add_argument('--nhead', type=int, default=8)
    parser.add_argument('--emb_size_c', type=int, default=128)
    parser.add_argument('--masking_ratio', type=float, default=0.01)
    parser.add_argument('--epochs', type=int, default=30)
    parser.add_argument('--ratio_highest_attention', type=float, default=0.35)
    #parser.add_argument('--dropout', type=float, default=0.01)
    parser.add_argument('--dropout', type=float, default=0.01)
    parser.add_argument('--nhid', type=int, default=128)
    parser.add_argument('--nhid_c', type=int, default=128)
    args = parser.parse_args(args=[])
    prop = utils.get_prop(args)
    
    prop['multi_group'] = [int(patch_index) for patch_index in prop['multi_group']]
    prop['batch_true'] = 1
    prop['nclasses'] = 7
    prop['seq_len'], prop['input_size'] = 2, 106
    prop['device'] = torch.device("cpu")
        
    
    
    model = MgMcFORMER(prop['multi_group'], prop['nclasses'], prop['seq_len'], prop['input_size'], prop['emb_size'], \
                       prop['nhid'], prop['emb_size_c'], prop['nhid_c'], prop['nhead'], prop['nlayers'], prop['device'],
                       prop['dropout']).to(torch.device("cpu"))
    
    
    model.load_state_dict(torch.load(model_name,map_location=torch.device('cpu')))
    
    model.eval()  
    
    return model,mean_1,std_1

def test_model(model, dataloader_test, nclasses, device):
    model.eval()  # Turn on the evaluation mode

    output_arr = []
    label_arr = []
    with torch.no_grad():
        for data, label in dataloader_test:
            data = data.to(device)
            
            label = label.to(device)
            #print(data[0])
            #print(model(data)[0])
            pred = model(data)[0]
            #print(model(data))
            
            output_arr.append(pred)
            label_arr.append(label)
            

    return label_arr,output_arr



def model_eval(final_data,model,mean,std):
   
    final_data = preprocess(final_data,mean,std)
    
    
    dataloader_train, dataloader_test = toDataloader_1(final_data,[0], final_data, [0])
    final_data = DataLoader(final_data,batch_size = 1)
    model.eval()
    
    label,output = test_model(model, dataloader_test, nclasses=8, device="cpu")
    probabilities = F.softmax(output[0],dim=1)
    
    return probabilities

def ML_Cks(
    filename=None,
    model=None,
    mean=None,
    std=None,
    model_type='normal',
    redshift=0.0,
    data_model='SDSS',
    base_dir="/share/SDSS/dr7/spec",
    tiled=None,
    loc=None,
    targetid=None
):
    if model_type == 'normal':
        final_data, med_snr = load_data(
            filename=filename,
            redshift=redshift,
            data_model=data_model,
            base_dir=base_dir,
            tiled=tiled,
            loc=loc,
            targetid=targetid
        )
    
    else:
        raise ValueError(f"unknown model_type: {model_type}")

    probabilities = model_eval(final_data, model, mean, std)
    
    return probabilities


def Get_Classification(
    filename=None,
    model_name=None,
    model_type='normal',
    redshift=0.0,
    data_model='SDSS_BIN',
    base_dir=None,
    tiled=None,
    loc=None,
    targetid=None
):
    
    model, mean, std = load_model(model_name, model_type='normal')

   
    final_data, med_snr = load_data(
        filename=filename,
        redshift=redshift,
        data_model=data_model,
        base_dir=base_dir,
        tiled=tiled,
        loc=loc,
        targetid=targetid
    )
    
  

    if final_data is None:
        raise ValueError(f"can't process: {filename}")


    probs = model_eval(final_data, model, mean, std)
    return probs



def classify_tensor(tensor, CLASS_NAMES=['TDE', 'Non-TDE'], verbose=True):
    if tensor.device.type != 'cpu':
        tensor = tensor.cpu()

    scores = tensor.squeeze().tolist()
    
    try:
        tde_index = CLASS_NAMES.index('TDE')
        tde_score = scores[tde_index]
    except ValueError:
        return "Error: 'TDE' not found in CLASS_NAMES"

    if verbose:
        
        return f"TDE Score: {tde_score:.4f}"
    else:
        
        return tde_score
    
    

In [13]:
# 1. Set public parameter
common_params = {
    'data_model': 'SDSS_BIN',
    'model_name': 'test_model_SP106-use4.pth',
    'model_type': 'normal'
}

# 2. test samples
test_cases = [
    # --- Real TDE ---
    
    ("./demo_data/spSpec-54562-1834-021.fit", 0.0415, "Real"),
    ("./demo_data/spSpec-53055-1737-369.fit", 0.0615328, "Real"),
    ("./demo_data/spSpec-52316-0601-226.fit", 0.0424, "Real"),
    
    
]

# 3. 统一执行
for path, z, tag in test_cases:
    if tag == "SEPARATOR":
        print(f"\n{'-'*21}\nFake TDE:\n")
        continue

    print(f"Testing: {path.split('/')[-1]}")
    
    probs = Get_Classification(filename=path, redshift=z, **common_params)
    
    
    print(classify_tensor(probs))
    print("-" * 30)

Testing: spSpec-54562-1834-021.fit
RA =  231.24879
DEC =  4.9064227
TDE Score: 0.9999
------------------------------
Testing: spSpec-53055-1737-369.fit
RA =  117.08614
DEC =  47.203973
TDE Score: 0.9833
------------------------------
Testing: spSpec-52316-0601-226.fit
RA =  190.60579
DEC =  64.488603
TDE Score: 0.7899
------------------------------


In [ ]:
import os
import sys
def run_all_subfolders(base_root, output_csv, model_path, model_type="normal", resume=True):

    try:
        for subfolder in sorted(os.listdir(base_root)):
            spSpec_dir = os.path.join(base_root, subfolder, "spSpec")
            if not os.path.isdir(spSpec_dir):
                continue 

            print(f"\n Processing {spSpec_dir}，result write in {output_csv}")

            run_sdss_batch_with_resume_v3(
                base_dir=spSpec_dir,
                output_csv=output_csv,
                model_path=model_path,
                model_type=model_type,
                resume=resume,
                
            )
    except KeyboardInterrupt:
        print("\n Kbd stop.")
        sys.exit(0)
        
        
def run_all_subfolders_seg(
    base_root, 
    output_csv, 
    model_path, 
    model_type="normal", 
    resume=True,
    n_segments=3,         
    segment_index=0       
):
    """
    Iterate through all subdirectories under base_root and execute 
    run_sdss_batch_with_resume_v3 for each spSpec folder. 
    All results are compiled into a single CSV file, supporting resume functionality.
    
    Added Parameters:
    - n_segments: Number of segments to split the subfolders into (default is 3).
    - segment_index: The specific segment to execute (0-based index).
    """
    try:
        
        subfolders = sorted([f for f in os.listdir(base_root) if os.path.isdir(os.path.join(base_root, f))])
        total_folders = len(subfolders)
        
        
        segment_size = math.ceil(total_folders / n_segments)
        start_idx = segment_index * segment_size
        end_idx = min(start_idx + segment_size, total_folders)
        
        
        subfolders_segment = subfolders[start_idx:end_idx]
        
        print(f"📂 {total_folders}")
        print(f"🔹 {segment_index+1}/{n_segments}: {len(subfolders_segment)} folders ({subfolders_segment[0]} ~ {subfolders_segment[-1]})")
        
        
        for subfolder in subfolders_segment:
            spSpec_dir = os.path.join(base_root, subfolder, "spSpec")
            if not os.path.isdir(spSpec_dir):
                continue  

            print(f"\n Processing {spSpec_dir}，write in {output_csv}")

            run_sdss_batch_with_resume_v3(
                base_dir=spSpec_dir,
                output_csv=output_csv,
                model_path=model_path,
                model_type=model_type,
                resume=resume,
            )
            
    except KeyboardInterrupt:
        print("\n⛔ kbd stop。")
        sys.exit(0)

In [ ]:
# The code 

run_all_subfolders_seg(
    base_root="/share/data/WideF-S/SDSS/dr7/spectro/ss_tar_26/",
    output_csv="./sdss_total_results_test_t14_PCA-0.csv",   
    model_path="test_model_SP106-use4.pth",
    model_type="normal",
    resume=True,
    segment_index=0
)